Esta es la libreta donde se realiza todo el flujo de trabajo para el RAG vectorial: desde importar los archivos del corpus indexado o las preguntas del banco de evaluación hasta los resultados generados por el modelo de lenguaje.

In [1]:
# Instalación de librerías
# Ejecutar al inicio de una sesión nueva de Colab.

!pip install -q faiss-cpu sentence-transformers pandas tqdm requests openpyxl python-docx


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 14.4 MB/s eta 0:00:00


In [2]:
# Cargar archivos manualmente
# Índice FAISS, chunks_filtrados, banco evaluación

from google.colab import files
import os

print("Selecciona los tres archivos necesarios:")
uploaded = files.upload()

print("\nArchivos cargados:")
for nombre in uploaded.keys():
    print("-", nombre)

Selecciona los tres archivos necesarios:


Saving banco_evaluacion_con_docs_rel.json to banco_evaluacion_con_docs_rel.json
Saving chunks_filtrados.json to chunks_filtrados.json
Saving indice_faiss_aml.index to indice_faiss_aml.index

Archivos cargados:
- banco_evaluacion_con_docs_rel.json
- chunks_filtrados.json
- indice_faiss_aml.index


In [3]:
import json

with open("banco_evaluacion_con_docs_rel.json", "r", encoding="utf-8") as f:
    banco = json.load(f)

print("Preguntas cargadas:", len(banco))
print("Q005:")
print(banco[4]["documentos_relacionados"])

Preguntas cargadas: 30
Q005:
[{'doc_id': '032', 'titulo': 'Lista consolidada sanciones UE'}, {'doc_id': '033', 'titulo': 'Lista consolidada sanciones ONU'}]


In [4]:
# Definir rutas de trabajo

import os

RUTA_BASE = "/content"

RUTA_INDICE = os.path.join(RUTA_BASE, "indice_faiss_aml.index")
RUTA_CHUNKS = os.path.join(RUTA_BASE, "chunks_filtrados.json")
RUTA_BANCO = os.path.join(RUTA_BASE, "banco_evaluacion_con_docs_rel.json")

RUTA_SALIDA_JSON = os.path.join(RUTA_BASE, "resultados_rag_vectorial.json")
RUTA_SALIDA_CSV = os.path.join(RUTA_BASE, "resultados_rag_vectorial.csv")

rutas_necesarias = [RUTA_INDICE, RUTA_CHUNKS, RUTA_BANCO]

for ruta in rutas_necesarias:
    print(ruta, "->", os.path.exists(ruta))

faltantes = [ruta for ruta in rutas_necesarias if not os.path.exists(ruta)]
if faltantes:
    raise FileNotFoundError(
        "Faltan archivos necesarios. Revisa que los nombres coincidan exactamente: "
        + ", ".join(faltantes)
    )


/content/indice_faiss_aml.index -> True
/content/chunks_filtrados.json -> True
/content/banco_evaluacion_con_docs_rel.json -> True


In [5]:
# Cargar índice, chunks y banco

import json
import faiss

indice = faiss.read_index(RUTA_INDICE)

with open(RUTA_CHUNKS, "r", encoding="utf-8") as f:
    chunks = json.load(f)

with open(RUTA_BANCO, "r", encoding="utf-8") as f:
    banco = json.load(f)

print("Vectores en FAISS:", indice.ntotal)
print("Chunks cargados:", len(chunks))
print("Preguntas cargadas:", len(banco))

if indice.ntotal != len(chunks):
    raise ValueError(
        f"El índice FAISS contiene {indice.ntotal} vectores, pero se han cargado {len(chunks)} chunks. "
        "Debes usar el índice y el JSON de chunks generados en la misma vectorización."
    )

if len(banco) == 0:
    raise ValueError("El banco de evaluación está vacío.")


Vectores en FAISS: 1529
Chunks cargados: 1529
Preguntas cargadas: 30


In [6]:
# Revisar estructura de los archivos

print("Ejemplo de chunk:")
print(json.dumps(chunks[0], ensure_ascii=False, indent=2)[:1500])

print("\nEjemplo de pregunta:")
print(json.dumps(banco[0], ensure_ascii=False, indent=2))

Ejemplo de chunk:
{
  "chunk_id": "001_0001",
  "doc_id": "001",
  "titulo": "Lavado de dinero",
  "categoria": "Tema central",
  "fuente": "Wikipedia",
  "idioma": "es",
  "tipo_documento": "entrada_enciclopedica",
  "formato_original": "html",
  "url": "https://es.wikipedia.org/wiki/Lavado_de_dinero",
  "texto_chunk": "El lavado de dinero (también conocido como lavado de capitales, lavado de activos, blanqueo de dinero,​ o blanqueo de capital) es el proceso de cambiar grandes cantidades de dinero obtenidas de delitos, como el narcotráfico, por otras de origen legítimo.​​ Es un delito en muchas jurisdicciones con definiciones diversas. Es una operación esencial del crimen organizado y de la economía sumergida. El blanqueo de capitales comienza con la comisión de un acto delictivo de tipo grave (tal como la obtención de beneficios ilegales en los mercados financieros u otros sectores económicos). El blanqueo de capitales es un delito autónomo que no requiere de una condena judicial pre

In [7]:
# Cargar modelo de embeddings E5

from sentence_transformers import SentenceTransformer
import numpy as np

MODELO_EMBEDDINGS = "intfloat/multilingual-e5-base"

modelo_embeddings = SentenceTransformer(MODELO_EMBEDDINGS)

print("Modelo cargado:", MODELO_EMBEDDINGS)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Modelo cargado: intfloat/multilingual-e5-base


In [8]:
# Función para vectorizar preguntas

def vectorizar_pregunta(pregunta):
    texto = "query: " + pregunta # nótese el prefijo query

    embedding = modelo_embeddings.encode(
        [texto],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    return embedding.astype("float32")

In [9]:
# Función de recuperación vectorial

def obtener_texto_chunk(chunk):
    posibles_claves = [
        "texto_chunk",
        "texto",
        "text",
        "content",
        "contenido"
    ]

    for clave in posibles_claves:
        if clave in chunk and chunk[clave]:
            return chunk[clave]

    return ""


def recuperar_chunks(pregunta, top_k=5):
    embedding_pregunta = vectorizar_pregunta(pregunta)

    scores, indices = indice.search(embedding_pregunta, top_k)

    resultados = []

    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        chunk = chunks[int(idx)]
        texto = obtener_texto_chunk(chunk)

        resultados.append({
            "rank": rank,
            "indice_chunk": int(idx),
            "score": float(score),
            "texto": texto,
            "metadatos": chunk
        })

    return resultados

In [10]:
# Probar recuperación con una pregunta

pregunta_prueba = banco[0].get("pregunta", banco[0].get("question", ""))

print("PREGUNTA:")
print(pregunta_prueba)

resultados_prueba = recuperar_chunks(pregunta_prueba, top_k=5)

for r in resultados_prueba:
    print("\n" + "="*80)
    print("Rank:", r["rank"])
    print("Score:", r["score"])
    print(r["texto"][:1000])

PREGUNTA:
¿Qué es el lavado de dinero?

Rank: 1
Score: 0.8584776520729065
El lavado de dinero (también conocido como lavado de capitales, lavado de activos, blanqueo de dinero,​ o blanqueo de capital) es el proceso de cambiar grandes cantidades de dinero obtenidas de delitos, como el narcotráfico, por otras de origen legítimo.​​ Es un delito en muchas jurisdicciones con definiciones diversas. Es una operación esencial del crimen organizado y de la economía sumergida. El blanqueo de capitales comienza con la comisión de un acto delictivo de tipo grave (tal como la obtención de beneficios ilegales en los mercados financieros u otros sectores económicos). El blanqueo de capitales es un delito autónomo que no requiere de una condena judicial previa por la comisión de la actividad delictiva por la que se originaron los fondos.​ Historia Hay autores que señalan que los delitos financieros, tanto la falsificación como el lavado de dinero, han existido incluso antes de que existiera el dinero 

In [11]:
# Instalar Ollama

!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 65 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (8,063 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama

In [19]:
# Iniciar servidor de Ollama

import subprocess
import time

proceso_ollama = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Servidor Ollama iniciado.")

Servidor Ollama iniciado.


In [20]:
# Descargar Llama 3.2 3B

!ollama pull llama3.2:3b

In [21]:
# Probar Ollama e incluir métricas de eficiencia

import requests

def consultar_ollama(
    prompt,
    modelo="llama3.2:3b",
    temperatura=0.0,
    devolver_metricas=False,
    num_ctx=8192,
    formato_json=False
):
    """
    Consulta un modelo local servido por Ollama.

    - devolver_metricas=True permite recoger conteos de tokens y duraciones.
    - num_ctx se fija explícitamente para evitar depender del contexto por defecto.
    - formato_json=True solicita a Ollama salida JSON, útil para el LLM como juez.
    """
    url = "http://localhost:11434/api/generate"

    payload = {
        "model": modelo,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperatura,
            "num_ctx": num_ctx
        }
    }

    if formato_json:
        payload["format"] = "json"

    respuesta = requests.post(url, json=payload, timeout=100)
    respuesta.raise_for_status()

    datos = respuesta.json()
    texto_generado = datos.get("response", "")

    metricas = {
        "prompt_eval_count": datos.get("prompt_eval_count"),
        "eval_count": datos.get("eval_count"),
        "total_duration": datos.get("total_duration"),
        "load_duration": datos.get("load_duration"),
        "prompt_eval_duration": datos.get("prompt_eval_duration"),
        "eval_duration": datos.get("eval_duration")
    }

    if devolver_metricas:
        return texto_generado, metricas

    return texto_generado


prueba = consultar_ollama("Responde en una frase: ¿qué significa RAG en recuperación aumentada por generación?")
print(prueba)


RAG (Recuperación Aumentada por Generación) es un método de recuperación que implica el uso de técnicas y herramientas específicas para ayudar a las personas a superar sus limitaciones físicas y emocionales, especialmente en contextos de trauma o lesión.


Más allá del resultado ofrecido de entrada, la prueba confirma que el servidor de Ollama responde correctamente.

In [22]:
# Prompt para usar con el sistema RAG

MAX_CARACTERES_CHUNK_GENERACION = 3500 # Para limitar la generación para que no se eternice, no el contexto previo.

def limitar_texto_generacion(texto, max_caracteres=MAX_CARACTERES_CHUNK_GENERACION):
    """Limita cada fragmento para controlar el tamaño del prompt de generación."""
    if texto is None:
        return ""
    texto = str(texto)
    if len(texto) <= max_caracteres:
        return texto
    return texto[:max_caracteres] + "\n[Fragmento truncado]"


def construir_prompt(pregunta, chunks_recuperados):
    contexto = ""

    for r in chunks_recuperados:
        metadatos = r.get("metadatos", {})
        doc_id = metadatos.get("doc_id", "")
        chunk_id = metadatos.get("chunk_id", "")
        titulo = metadatos.get("titulo", "")
        fuente = metadatos.get("fuente", "")
        url = metadatos.get("url", "")

        contexto += f"\n[Fragmento {r['rank']} | score={r['score']:.4f} | doc_id={doc_id} | chunk_id={chunk_id}]\n"
        if titulo or fuente:
            contexto += f"Título: {titulo}\nFuente: {fuente}\n"
        if url:
            contexto += f"URL: {url}\n"
        contexto += limitar_texto_generacion(r["texto"])
        contexto += "\n"

    prompt = f"""
Eres un asistente especializado en prevención del blanqueo de capitales y análisis documental.

Responde a la pregunta usando únicamente la información del contexto proporcionado.
No utilices conocimiento externo.
Si el contexto no contiene información suficiente para responder, indícalo claramente.
Redacta la respuesta en español, de forma clara y precisa.
No inventes normas, organismos, relaciones ni ejemplos que no aparezcan en el contexto.

CONTEXTO:
{contexto}

PREGUNTA:
{pregunta}

RESPUESTA:
"""

    return prompt.strip()


In [23]:
# Ejecutar RAG vectorial sobre todo el banco de evaluación con métricas de eficiencia

from tqdm import tqdm
import time

TOP_K = 5
MODELO_GENERATIVO = "llama3.2:3b"
NUM_CTX_GENERACION = 8192

resultados_rag = []

for item in tqdm(banco):
    pregunta = item.get("pregunta", item.get("question", ""))
    id_pregunta = item.get("id", "")

    inicio_total = time.time()

    inicio_recuperacion = time.time()
    chunks_recuperados = recuperar_chunks(pregunta, top_k=TOP_K)
    fin_recuperacion = time.time()

    prompt = construir_prompt(pregunta, chunks_recuperados)

    inicio_generacion = time.time()
    respuesta_generada, metricas_ollama = consultar_ollama(
        prompt,
        modelo=MODELO_GENERATIVO,
        temperatura=0.0,
        devolver_metricas=True,
        num_ctx=NUM_CTX_GENERACION
    )
    fin_generacion = time.time()

    fin_total = time.time()

    resultado = {
        "id": id_pregunta,
        "pregunta": pregunta,
        "datos_banco": item,
        "top_k": TOP_K,
        "modelo_embeddings": MODELO_EMBEDDINGS,
        "modelo_generativo": MODELO_GENERATIVO,
        "num_ctx_generacion": NUM_CTX_GENERACION,
        "chunks_recuperados": chunks_recuperados,
        "respuesta_generada": respuesta_generada,

        # Latencia total del sistema RAG: recuperación + construcción del prompt + generación.
        "latencia_segundos": fin_total - inicio_total, # La principal, es total, suma de las siguientes
        "latencia_recuperacion_segundos": fin_recuperacion - inicio_recuperacion, # Para latencia de recuperación
        "latencia_generacion_segundos": fin_generacion - inicio_generacion, # Para latencia de generación

        # Métricas devueltas por Ollama.
        "tokens_entrada": metricas_ollama.get("prompt_eval_count"),
        "tokens_salida": metricas_ollama.get("eval_count"),
        "duracion_total_s": round(metricas_ollama.get("total_duration", 0) / 1e9, 3), # Para que se muestre en segundos y no en ns
        "duracion_carga_s": round(metricas_ollama.get("load_duration", 0) / 1e9, 3),
        "duracion_prompt_s": round(metricas_ollama.get("prompt_eval_duration", 0) / 1e9, 3),
        "duracion_generacion_s": round(metricas_ollama.get("eval_duration", 0) / 1e9, 3),

        # Control adicional para detectar prompts excesivos.
        "longitud_prompt_caracteres": len(prompt)
    }

    resultados_rag.append(resultado)

print("Preguntas procesadas:", len(resultados_rag))


100%|██████████| 30/30 [04:00<00:00,  8.02s/it]

Preguntas procesadas: 30


In [24]:
# Guardar resultados en JSON base

with open(RUTA_SALIDA_JSON, "w", encoding="utf-8") as f:
    json.dump(resultados_rag, f, ensure_ascii=False, indent=2)

print("Archivo JSON guardado en:")
print(RUTA_SALIDA_JSON)

Archivo JSON guardado en:
/content/resultados_rag_vectorial.json


In [25]:
# Funciones auxiliares para la evaluación automática mediante LLM como juez
# Inspirada en RAGAS, pero integrada en un único procedimiento de evaluación.

import re
import json

def extraer_json(texto):
    """
    Extrae un objeto JSON de la salida del modelo evaluador.
    La función contempla el caso ideal, en el que el modelo devuelve únicamente JSON,
    y el caso frecuente en el que añade texto antes o después.
    """
    try:
        return json.loads(texto)
    except Exception:
        pass

    patron = r"\{[\s\S]*\}"
    coincidencia = re.search(patron, texto)

    if coincidencia:
        try:
            return json.loads(coincidencia.group(0))
        except Exception:
            return None

    return None


def limitar_texto(texto, max_caracteres=1800):
    """
    Limita la longitud de cada fragmento para evitar prompts de evaluación demasiado extensos.
    """
    if texto is None:
        return ""

    texto = str(texto)

    if len(texto) <= max_caracteres:
        return texto

    return texto[:max_caracteres] + "\n[Texto truncado]"


def normalizar_puntuacion(valor):
    """
    Convierte una puntuación a float, la limita al intervalo 0.0-1.0
    y la redondea a un decimal, de acuerdo con la escala definida en la memoria.
    """
    try:
        valor = float(valor)
    except Exception:
        return None

    valor = max(0.0, min(1.0, valor))
    return round(valor, 1)


def normalizar_evaluacion_llm(datos, salida_original=None):
    """
    Normaliza la salida del evaluador para asegurar que todas las dimensiones
    estén presentes con los nombres empleados en el diseño de evaluación.
    """
    if not isinstance(datos, dict):
        return {
            "relevancia_contexto": None,
            "relevancia_respuesta": None,
            "fidelidad_respuesta_contexto": None,
            "justificacion_relevancia_contexto": "",
            "justificacion_relevancia_respuesta": "",
            "justificacion_fidelidad_respuesta_contexto": "",
            "justificacion_breve": "No se pudo extraer un JSON válido.",
            "salida_original": salida_original
        }

    evaluacion = {
        "relevancia_contexto": normalizar_puntuacion(datos.get("relevancia_contexto")),
        "relevancia_respuesta": normalizar_puntuacion(datos.get("relevancia_respuesta")),
        "fidelidad_respuesta_contexto": normalizar_puntuacion(datos.get("fidelidad_respuesta_contexto")),
        "justificacion_relevancia_contexto": str(datos.get("justificacion_relevancia_contexto", "")),
        "justificacion_relevancia_respuesta": str(datos.get("justificacion_relevancia_respuesta", "")),
        "justificacion_fidelidad_respuesta_contexto": str(datos.get("justificacion_fidelidad_respuesta_contexto", "")),
        "justificacion_breve": str(datos.get("justificacion_breve", ""))
    }

    puntuaciones = [
        evaluacion["relevancia_contexto"],
        evaluacion["relevancia_respuesta"],
        evaluacion["fidelidad_respuesta_contexto"]
    ]

    puntuaciones_validas = [p for p in puntuaciones if p is not None]

    if puntuaciones_validas:
        evaluacion["media_evaluacion_automatica"] = round(
            sum(puntuaciones_validas) / len(puntuaciones_validas),
            3
        )
    else:
        evaluacion["media_evaluacion_automatica"] = None

    return evaluacion


In [26]:
# Prompt y función de evaluación automática mediante LLM como juez
# Diseño: role-play + few-shot + razonamiento paso a paso interno + escala explícita.

NUM_CTX_EVALUADOR = 8192

def construir_prompt_evaluador_llm(pregunta, respuesta, chunks_recuperados):
    contexto = ""

    for r in chunks_recuperados:
        texto_chunk = limitar_texto(r.get("texto", ""), max_caracteres=1800)
        metadatos = r.get("metadatos", {})
        doc_id = metadatos.get("doc_id", "")
        chunk_id = metadatos.get("chunk_id", "")
        titulo = metadatos.get("titulo", "")

        contexto += f"\n[Fragmento {r.get('rank', '')} | doc_id={doc_id} | chunk_id={chunk_id} | título={titulo}]\n"
        contexto += texto_chunk
        contexto += "\n"

    prompt_eval = f"""
Actúa como un evaluador externo experto en sistemas RAG aplicados a un dominio especializado.
Tu tarea es evaluar una respuesta generada por un sistema RAG a partir de una pregunta y de un contexto recuperado.

Esta evaluación está inspirada en RAGAS, pero se formula como una única evaluación automática mediante LLM como juez.
No estás aplicando RAGAS como paquete independiente: estás aplicando tres dimensiones inspiradas en RAGAS dentro de un mismo procedimiento.

Debes valorar tres dimensiones:

1. relevancia_contexto
Nombre conceptual en la memoria: relevancia del contexto recuperado.
Evalúa si el contexto recuperado es útil para responder a la pregunta.
- 0.0: el contexto recuperado es irrelevante o inútil para responder a la pregunta.
- 0.1-0.3: el contexto recuperado es muy poco útil, aunque pueda contener alguna relación superficial con el tema.
- 0.4-0.6: el contexto es parcialmente útil, pero resulta incompleto, impreciso o contiene ruido.
- 0.7-0.8: el contexto es bastante pertinente y útil, aunque no sea perfecto o pueda faltar alguna información.
- 0.9-1.0: el contexto es muy pertinente y claramente útil para responder a la pregunta.

2. relevancia_respuesta
Nombre conceptual en la memoria: relevancia de la respuesta.
Evalúa si la respuesta generada responde de forma adecuada a la pregunta planteada.
- 0.0: la respuesta no responde a la pregunta o responde de forma incorrecta.
- 0.1-0.3: la respuesta es muy deficiente, demasiado vaga o apenas relacionada con la consulta.
- 0.4-0.6: la respuesta es parcialmente adecuada, pero incompleta, poco precisa o con limitaciones importantes.
- 0.7-0.8: la respuesta es buena y responde a la pregunta, aunque presenta pequeños fallos, imprecisiones u omisiones.
- 0.9-1.0: la respuesta es muy buena o prácticamente perfecta, correcta, suficiente y adecuada al dominio.

3. fidelidad_respuesta_contexto
Nombre conceptual en la memoria: fidelidad de la respuesta al contexto.
Evalúa si las afirmaciones de la respuesta están apoyadas por el contexto recuperado.
No valores aquí si la respuesta es completa; valora solo si lo que afirma está fundamentado en el contexto.
- 0.0: la respuesta no está apoyada en el contexto recuperado o contiene información claramente no fundamentada.
- 0.1-0.3: la respuesta está muy poco apoyada en el contexto o incluye bastantes afirmaciones dudosas.
- 0.4-0.6: la respuesta está parcialmente apoyada en el contexto, pero mezcla información fundamentada con afirmaciones poco claras o no suficientemente respaldadas.
- 0.7-0.8: la respuesta está mayoritariamente apoyada en el contexto, aunque puede incluir alguna afirmación menor no del todo justificada.
- 0.9-1.0: la respuesta está plenamente o casi plenamente apoyada en el contexto recuperado.

Instrucciones de razonamiento:
- Analiza el caso paso a paso internamente antes de puntuar.
- No muestres tu razonamiento paso a paso.
- Distingue con cuidado entre las tres dimensiones.
- Una respuesta puede ser fiel al contexto pero incompleta: en ese caso, baja la relevancia_respuesta, no necesariamente la fidelidad.
- Un contexto puede ser relevante aunque la respuesta generada sea mala.
- Usa puntuaciones de 0.0 a 1.0 con un decimal.
- Devuelve únicamente un JSON válido. No añadas texto fuera del JSON.

Ejemplo 1:
PREGUNTA:
¿Qué es la diligencia debida del cliente?

CONTEXTO RECUPERADO:
[Fragmento 1]
La diligencia debida del cliente comprende la identificación del cliente, la verificación de su identidad y el seguimiento continuo de la relación de negocios.

RESPUESTA GENERADA:
La diligencia debida del cliente consiste en identificar al cliente, verificar su identidad y realizar un seguimiento continuo de la relación de negocios.

JSON:
{{
  "relevancia_contexto": 1.0,
  "relevancia_respuesta": 1.0,
  "fidelidad_respuesta_contexto": 1.0,
  "justificacion_relevancia_contexto": "El contexto contiene directamente la información necesaria para responder.",
  "justificacion_relevancia_respuesta": "La respuesta responde de forma directa, suficiente y precisa a la pregunta.",
  "justificacion_fidelidad_respuesta_contexto": "Todas las afirmaciones están apoyadas por el contexto recuperado.",
  "justificacion_breve": "Caso correcto: buen contexto, buena respuesta y alta fidelidad."
}}

Ejemplo 2:
PREGUNTA:
¿Qué medidas pueden adoptarse para gestionar clientes de alto riesgo sin cerrar automáticamente sus cuentas?

CONTEXTO RECUPERADO:
[Fragmento 1]
Las entidades deben aplicar medidas reforzadas cuando el riesgo sea elevado y mantener controles proporcionales al perfil del cliente.

RESPUESTA GENERADA:
Las entidades deben cerrar siempre las cuentas de los clientes de alto riesgo para evitar sanciones.

JSON:
{{
  "relevancia_contexto": 0.8,
  "relevancia_respuesta": 0.0,
  "fidelidad_respuesta_contexto": 0.0,
  "justificacion_relevancia_contexto": "El contexto es pertinente porque menciona medidas reforzadas y controles proporcionales.",
  "justificacion_relevancia_respuesta": "La respuesta contradice la orientación de la pregunta y propone el cierre automático.",
  "justificacion_fidelidad_respuesta_contexto": "La afirmación de cerrar siempre las cuentas no está apoyada por el contexto.",
  "justificacion_breve": "El contexto es útil, pero la respuesta es incorrecta y no fiel."
}}

Ejemplo 3:
PREGUNTA:
¿Qué factores deben tenerse en cuenta en un enfoque basado en el riesgo?

CONTEXTO RECUPERADO:
[Fragmento 1]
El enfoque basado en el riesgo exige considerar factores como el tipo de cliente, el producto, el canal de distribución y la zona geográfica.

RESPUESTA GENERADA:
El enfoque basado en el riesgo exige analizar el tipo de cliente y el producto.

JSON:
{{
  "relevancia_contexto": 1.0,
  "relevancia_respuesta": 0.6,
  "fidelidad_respuesta_contexto": 1.0,
  "justificacion_relevancia_contexto": "El contexto recuperado contiene todos los factores necesarios.",
  "justificacion_relevancia_respuesta": "La respuesta es correcta pero incompleta, porque omite el canal de distribución y la zona geográfica.",
  "justificacion_fidelidad_respuesta_contexto": "Las afirmaciones de la respuesta están apoyadas por el contexto.",
  "justificacion_breve": "La respuesta es fiel, pero solo parcialmente completa."
}}

Ahora evalúa el siguiente caso real.

PREGUNTA:
{pregunta}

CONTEXTO RECUPERADO:
{contexto}

RESPUESTA GENERADA:
{respuesta}

JSON:
"""

    return prompt_eval


def evaluar_llm_como_juez(pregunta, respuesta, chunks_recuperados, modelo_evaluador=None):
    """
    Evalúa automáticamente un caso mediante un LLM como juez.
    El modelo recibe la pregunta, el contexto recuperado y la respuesta generada.
    """
    if modelo_evaluador is None:
        modelo_evaluador = MODELO_GENERATIVO

    prompt_eval = construir_prompt_evaluador_llm(
        pregunta=pregunta,
        respuesta=respuesta,
        chunks_recuperados=chunks_recuperados
    )

    salida = consultar_ollama(
        prompt_eval,
        modelo=modelo_evaluador,
        temperatura=0.0,
        num_ctx=NUM_CTX_EVALUADOR,
        formato_json=True
    )

    datos = extraer_json(salida)

    return normalizar_evaluacion_llm(datos, salida_original=salida)


In [27]:
# Ejecutar evaluación automática mediante LLM como juez inspirada en RAGAS

MODELO_EVALUADOR = MODELO_GENERATIVO

for r in tqdm(resultados_rag):
    evaluacion = evaluar_llm_como_juez(
        pregunta=r["pregunta"],
        respuesta=r["respuesta_generada"],
        chunks_recuperados=r["chunks_recuperados"],
        modelo_evaluador=MODELO_EVALUADOR
    )

    r["modelo_evaluador"] = MODELO_EVALUADOR
    r["num_ctx_evaluador"] = NUM_CTX_EVALUADOR

    r["eval_relevancia_contexto"] = evaluacion.get("relevancia_contexto")
    r["eval_relevancia_respuesta"] = evaluacion.get("relevancia_respuesta")
    r["eval_fidelidad_respuesta_contexto"] = evaluacion.get("fidelidad_respuesta_contexto")
    r["eval_media_automatica"] = evaluacion.get("media_evaluacion_automatica")

    r["eval_justificacion_relevancia_contexto"] = evaluacion.get("justificacion_relevancia_contexto", "")
    r["eval_justificacion_relevancia_respuesta"] = evaluacion.get("justificacion_relevancia_respuesta", "")
    r["eval_justificacion_fidelidad_respuesta_contexto"] = evaluacion.get("justificacion_fidelidad_respuesta_contexto", "")
    r["eval_justificacion_breve"] = evaluacion.get("justificacion_breve", "")

    if "salida_original" in evaluacion:
        r["eval_salida_original"] = evaluacion["salida_original"]

print("Evaluación automática mediante LLM como juez completada.")


100%|██████████| 30/30 [02:24<00:00,  4.81s/it]

Evaluación automática mediante LLM como juez completada.


In [28]:
# Guardar JSON final con métricas y evaluación automática

RUTA_SALIDA_JSON_METRICAS = os.path.join(
    RUTA_BASE,
    "resultados_rag_vectorial_con_evaluacion_automatica.json"
)

with open(RUTA_SALIDA_JSON_METRICAS, "w", encoding="utf-8") as f:
    json.dump(resultados_rag, f, ensure_ascii=False, indent=2)

print("JSON con métricas y evaluación automática guardado en:")
print(RUTA_SALIDA_JSON_METRICAS)


JSON con métricas y evaluación automática guardado en:
/content/resultados_rag_vectorial_con_evaluacion_automatica.json


In [29]:
# Crear CSV final con métricas de eficiencia y evaluación automática

import pandas as pd

filas = []

for r in resultados_rag:
    fila = {
        "id": r["id"],
        "pregunta": r["pregunta"],
        "respuesta_generada": r["respuesta_generada"],

        "top_k": r["top_k"],
        "modelo_embeddings": r["modelo_embeddings"],
        "modelo_generativo": r["modelo_generativo"],
        "modelo_evaluador": r.get("modelo_evaluador"),
        "num_ctx_generacion": r.get("num_ctx_generacion"),
        "num_ctx_evaluador": r.get("num_ctx_evaluador"),

        "latencia_segundos": r.get("latencia_segundos"),
        "latencia_recuperacion_segundos": r.get("latencia_recuperacion_segundos"),
        "latencia_generacion_segundos": r.get("latencia_generacion_segundos"),
        "tokens_entrada": r.get("tokens_entrada"),
        "tokens_salida": r.get("tokens_salida"),
        "longitud_prompt_caracteres": r.get("longitud_prompt_caracteres"),

        "duracion_total_ns": r.get("duracion_total_ns"),
        "duracion_carga_ns": r.get("duracion_carga_ns"),
        "duracion_prompt_ns": r.get("duracion_prompt_ns"),
        "duracion_generacion_ns": r.get("duracion_generacion_ns"),

        "eval_relevancia_contexto": r.get("eval_relevancia_contexto"),
        "eval_relevancia_respuesta": r.get("eval_relevancia_respuesta"),
        "eval_fidelidad_respuesta_contexto": r.get("eval_fidelidad_respuesta_contexto"),
        "eval_media_automatica": r.get("eval_media_automatica"),

        "eval_justificacion_relevancia_contexto": r.get("eval_justificacion_relevancia_contexto"),
        "eval_justificacion_relevancia_respuesta": r.get("eval_justificacion_relevancia_respuesta"),
        "eval_justificacion_fidelidad_respuesta_contexto": r.get("eval_justificacion_fidelidad_respuesta_contexto"),
        "eval_justificacion_breve": r.get("eval_justificacion_breve")
    }

    datos_banco = r.get("datos_banco", {})

    for clave, valor in datos_banco.items():
        if clave not in fila:
            if isinstance(valor, (dict, list)):
                fila[f"banco_{clave}"] = json.dumps(valor, ensure_ascii=False)
            else:
                fila[f"banco_{clave}"] = valor

    for chunk in r["chunks_recuperados"]:
        n = chunk["rank"]
        metadatos = chunk.get("metadatos", {})
        fila[f"chunk_{n}_score"] = chunk["score"]
        fila[f"chunk_{n}_texto"] = chunk["texto"]
        fila[f"chunk_{n}_indice"] = chunk["indice_chunk"]
        fila[f"chunk_{n}_doc_id"] = metadatos.get("doc_id", "")
        fila[f"chunk_{n}_chunk_id"] = metadatos.get("chunk_id", "")
        fila[f"chunk_{n}_titulo"] = metadatos.get("titulo", "")
        fila[f"chunk_{n}_fuente"] = metadatos.get("fuente", "")

    filas.append(fila)

df_resultados_metricas = pd.DataFrame(filas)

RUTA_SALIDA_CSV_METRICAS = os.path.join(
    RUTA_BASE,
    "resultados_rag_vectorial_con_evaluacion_automatica.csv"
)

df_resultados_metricas.to_csv(
    RUTA_SALIDA_CSV_METRICAS,
    index=False,
    encoding="utf-8-sig"
)

print("CSV con métricas y evaluación automática guardado en:")
print(RUTA_SALIDA_CSV_METRICAS)

df_resultados_metricas.head()


CSV con métricas y evaluación automática guardado en:
/content/resultados_rag_vectorial_con_evaluacion_automatica.csv


,id,pregunta,respuesta_generada,top_k,modelo_embeddings,modelo_generativo,modelo_evaluador,num_ctx_generacion,num_ctx_evaluador,latencia_segundos,...,chunk_4_chunk_id,chunk_4_titulo,chunk_4_fuente,chunk_5_score,chunk_5_texto,chunk_5_indice,chunk_5_doc_id,chunk_5_chunk_id,chunk_5_titulo,chunk_5_fuente
0,Q001,¿Qué es el lavado de dinero?,El lavado de dinero es el proceso de cambiar g...,5,intfloat/multilingual-e5-base,llama3.2:3b,llama3.2:3b,8192,8192,4.492678,...,001_0002,Lavado de dinero,Wikipedia,0.844934,objetivos a combatir la financiación del terro...,4,001,001_0005,Lavado de dinero,Wikipedia
1,Q002,¿Qué es el pitufeo?,El pitufeo es una práctica de lavado de activo...,5,intfloat/multilingual-e5-base,llama3.2:3b,llama3.2:3b,8192,8192,3.724740,...,013_0001,Testaferro,Wikipedia,0.768415,es su impacto negativo en la innovación de pro...,40,015,015_0002,Corrupción empresarial,Wikipedia
2,Q003,¿Qué significa persona expuesta políticamente?,"Según el contexto proporcionado, una Persona E...",5,intfloat/multilingual-e5-base,llama3.2:3b,llama3.2:3b,8192,8192,5.075552,...,023_0001,Conozca a su cliente,Wikipedia,0.786803,"El término testaferro se usa en la literatura,...",37,013,013_0001,Testaferro,Wikipedia
3,Q004,¿Qué es la diligencia debida?,La diligencia debida se refiere a la investiga...,5,intfloat/multilingual-e5-base,llama3.2:3b,llama3.2:3b,8192,8192,5.055417,...,093_0003,Catálogo indicador de riesgo sectores de entid...,Tesoro_Público,0.825534,real que regularmente aparece en constitucione...,1279,100,100_0006,Catálogo indicador de riesgo sectores de entid...,Tesoro_Público
4,Q005,La entidad Amroggang Development Banking Corpo...,"Lo siento, pero no tengo información suficient...",5,intfloat/multilingual-e5-base,llama3.2:3b,llama3.2:3b,8192,8192,4.188197,...,090_0196,Estándares internacionales sobre la lucha cont...,GAFILAT,0.816021,"10/2010, de 28 de abril, de prevención del bla...",399,035,035_0003,Comunicación informativa sobre aplicación sanc...,Tesoro_Público


In [30]:
# Resumen de métricas principales

metricas_resumen = [
    "latencia_segundos",
    "latencia_recuperacion_segundos",
    "latencia_generacion_segundos",
    "tokens_entrada",
    "tokens_salida",
    "longitud_prompt_caracteres",
    "eval_relevancia_contexto",
    "eval_relevancia_respuesta",
    "eval_fidelidad_respuesta_contexto",
    "eval_media_automatica"
]

resumen_metricas = df_resultados_metricas[metricas_resumen].mean(numeric_only=True)

print("Resumen medio de métricas:")
print(resumen_metricas)


Resumen medio de métricas:
latencia_segundos                        8.018859
latencia_recuperacion_segundos           0.017653
latencia_generacion_segundos             8.001136
tokens_entrada                        4660.600000
tokens_salida                          277.266667
longitud_prompt_caracteres           17723.566667
eval_relevancia_contexto                 0.823333
eval_relevancia_respuesta                0.676667
eval_fidelidad_respuesta_contexto        0.753333
eval_media_automatica                    0.751233
dtype: float64


In [31]:
# Crear una tabla resumida con únicamente las métricas principales

columnas_resumen = [
    "id",
    "pregunta",
    "latencia_segundos",
    "latencia_recuperacion_segundos",
    "latencia_generacion_segundos",
    "tokens_entrada",
    "tokens_salida",
    "longitud_prompt_caracteres",
    "eval_relevancia_contexto",
    "eval_relevancia_respuesta",
    "eval_fidelidad_respuesta_contexto",
    "eval_media_automatica"
]

df_metricas_resumidas = df_resultados_metricas[columnas_resumen].copy()

# Redondear las métricas para facilitar la lectura
columnas_numericas = [
    "latencia_segundos",
    "latencia_recuperacion_segundos",
    "latencia_generacion_segundos",
    "tokens_entrada",
    "tokens_salida",
    "longitud_prompt_caracteres",
    "eval_relevancia_contexto",
    "eval_relevancia_respuesta",
    "eval_fidelidad_respuesta_contexto",
    "eval_media_automatica"
]

df_metricas_resumidas[columnas_numericas] = (
    df_metricas_resumidas[columnas_numericas].round(4)
)

# Guardar archivo resumido
RUTA_SALIDA_CSV_RESUMIDO = os.path.join(
    RUTA_BASE,
    "metricas_resumidas_por_pregunta_rag_vectorial.csv"
)

df_metricas_resumidas.to_csv(
    RUTA_SALIDA_CSV_RESUMIDO,
    index=False,
    encoding="utf-8-sig"
)

print("Tabla resumida guardada en:")
print(RUTA_SALIDA_CSV_RESUMIDO)

df_metricas_resumidas.head(10)


Tabla resumida guardada en:
/content/metricas_resumidas_por_pregunta_rag_vectorial.csv


,id,pregunta,latencia_segundos,latencia_recuperacion_segundos,latencia_generacion_segundos,tokens_entrada,tokens_salida,longitud_prompt_caracteres,eval_relevancia_contexto,eval_relevancia_respuesta,eval_fidelidad_respuesta_contexto,eval_media_automatica
0,Q001,¿Qué es el lavado de dinero?,4.4927,0.0151,4.4775,4502,121,17383,0.9,1.0,1.0,0.967
1,Q002,¿Qué es el pitufeo?,3.7247,0.0217,3.7030,3734,80,14103,1.0,1.0,1.0,1.000
2,Q003,¿Qué significa persona expuesta políticamente?,5.0756,0.0176,5.0578,4689,90,18127,1.0,1.0,1.0,1.000
3,Q004,¿Qué es la diligencia debida?,5.0554,0.0149,5.0404,4313,139,15594,0.9,1.0,1.0,0.967
4,Q005,La entidad Amroggang Development Banking Corpo...,4.1882,0.0269,4.1612,4364,82,16399,0.2,0.0,0.0,0.067
5,Q006,¿Cuál es la diferencia entre evasión fiscal y ...,6.6960,0.0143,6.6816,4267,237,15921,0.9,0.8,0.7,0.800
6,Q007,¿En qué se diferencian una empresa fantasma y ...,7.5263,0.0138,7.5124,4077,284,15977,0.9,0.8,0.7,0.800
7,Q008,¿Qué diferencia existe entre dinero negro y ec...,4.0422,0.0149,4.0273,4354,69,16847,0.2,0.0,0.0,0.067
8,Q009,¿Qué relación existe entre las personas expues...,5.1393,0.0158,5.1235,4507,124,14539,0.9,0.8,0.9,0.867
9,Q010,¿Cómo se relaciona la propiedad efectiva con e...,9.8037,0.0201,9.7836,4233,363,16953,0.9,0.8,0.7,0.800


A continuación, se procede a incluir algunas celdas para obtener las preguntas generadas a las respuestas del banco de manera más limpia para que la experta que actúa de jueza pueda valorarlas más limpia y cómodamente.

In [32]:
# Crear un archivo sencillo para la evaluación de la experta
# Incluye identificador, pregunta, respuesta generada, puntuación cuantitativa y comentarios.

import pandas as pd
import os

filas_experta = []

for r in resultados_rag:
    datos_banco = r.get("datos_banco", {})
    filas_experta.append({
        "id": r["id"],
        "pregunta": r["pregunta"],
        "tipo_pregunta": datos_banco.get("tipo_pregunta", ""),
        "complejidad_razonamiento": datos_banco.get("complejidad_razonamiento", ""),
        "distribucion_evidencia": datos_banco.get("distribucion_evidencia", ""),
        "respuesta_generada": r["respuesta_generada"],
        "puntuacion_relevancia_respuesta_experta_0_1": "",
        "comentarios_experta": ""
    })

df_experta = pd.DataFrame(filas_experta)

escala_experta = pd.DataFrame([
    {"tramo": "0,0", "interpretacion": "La respuesta no responde a la pregunta o responde de forma incorrecta."},
    {"tramo": "0,1-0,3", "interpretacion": "La respuesta es muy deficiente, demasiado vaga o apenas relacionada con la consulta."},
    {"tramo": "0,4-0,6", "interpretacion": "La respuesta es parcialmente adecuada, pero incompleta, poco precisa o con limitaciones importantes."},
    {"tramo": "0,7-0,8", "interpretacion": "La respuesta es buena y responde a la pregunta, aunque presenta pequeños fallos, imprecisiones u omisiones."},
    {"tramo": "0,9-1,0", "interpretacion": "La respuesta es muy buena o prácticamente perfecta, correcta, suficiente y adecuada al dominio."},
])

# Guardar en Excel para facilitar la revisión y anotación manual
RUTA_EVALUACION_EXPERTA = os.path.join(
    RUTA_BASE,
    "evaluacion_experta_rag_vectorial.xlsx"
)

with pd.ExcelWriter(RUTA_EVALUACION_EXPERTA, engine="openpyxl") as writer:
    df_experta.to_excel(
        writer,
        sheet_name="Evaluacion",
        index=False
    )
    escala_experta.to_excel(
        writer,
        sheet_name="Escala",
        index=False
    )

print("Archivo para la experta guardado en:")
print(RUTA_EVALUACION_EXPERTA)

# Mostrar una vista previa
df_experta.head(10)


Archivo para la experta guardado en:
/content/evaluacion_experta_rag_vectorial.xlsx


,id,pregunta,tipo_pregunta,complejidad_razonamiento,distribucion_evidencia,respuesta_generada,puntuacion_relevancia_respuesta_experta_0_1,comentarios_experta
0,Q001,¿Qué es el lavado de dinero?,factual simple,single-hop,documento único,El lavado de dinero es el proceso de cambiar g...,,
1,Q002,¿Qué es el pitufeo?,factual simple,single-hop,documento único,El pitufeo es una práctica de lavado de activo...,,
2,Q003,¿Qué significa persona expuesta políticamente?,factual simple,single-hop,documento único,"Según el contexto proporcionado, una Persona E...",,
3,Q004,¿Qué es la diligencia debida?,factual simple,single-hop,documento único,La diligencia debida se refiere a la investiga...,,
4,Q005,La entidad Amroggang Development Banking Corpo...,factual simple,single-hop,múltiples documentos,"Lo siento, pero no tengo información suficient...",,
5,Q006,¿Cuál es la diferencia entre evasión fiscal y ...,comparativa,single-hop,múltiples documentos,La diferencia entre evasión fiscal y elusión f...,,
6,Q007,¿En qué se diferencian una empresa fantasma y ...,comparativa,single-hop,múltiples documentos,Una empresa fantasma y una organización fachad...,,
7,Q008,¿Qué diferencia existe entre dinero negro y ec...,comparativa,single-hop,múltiples documentos,La pregunta no se puede responder con la infor...,,
8,Q009,¿Qué relación existe entre las personas expues...,relacional,multi-hop,múltiples documentos,"Según el fragmento 4, cuando un cliente o un t...",,
9,Q010,¿Cómo se relaciona la propiedad efectiva con e...,relacional,multi-hop,múltiples documentos,La propiedad efectiva está estrechamente relac...,,


In [33]:
from docx import Document
from docx.shared import Pt
import os

RUTA_WORD_EXPERTA = os.path.join(
    RUTA_BASE,
    "evaluacion_experta_rag_vectorial.docx"
)

documento = Document()

documento.add_heading("Evaluación experta del sistema RAG vectorial", level=1)

documento.add_paragraph(
    "Este documento recoge las preguntas del banco de evaluación y las respuestas generadas "
    "por el sistema RAG vectorial. La experta valorará la relevancia de la respuesta, es decir, "
    "si la respuesta responde de forma correcta, suficiente y adecuada a la pregunta planteada. "
    "La puntuación debe expresarse entre 0,0 y 1,0, con un decimal."
)

documento.add_heading("Escala de relevancia de la respuesta", level=2)
documento.add_paragraph("0,0: la respuesta no responde a la pregunta o responde de forma incorrecta.")
documento.add_paragraph("0,1-0,3: la respuesta es muy deficiente, demasiado vaga o apenas relacionada con la consulta.")
documento.add_paragraph("0,4-0,6: la respuesta es parcialmente adecuada, pero incompleta, poco precisa o con limitaciones importantes.")
documento.add_paragraph("0,7-0,8: la respuesta es buena y responde a la pregunta, aunque presenta pequeños fallos, imprecisiones u omisiones.")
documento.add_paragraph("0,9-1,0: la respuesta es muy buena o prácticamente perfecta, correcta, suficiente y adecuada al dominio.")

def valor_seguro(fila, columna):
    return str(fila[columna]) if columna in fila and pd.notna(fila[columna]) else ""

for _, fila in df_experta.iterrows():
    documento.add_heading(f"{fila['id']}. {fila['pregunta']}", level=2)

    if valor_seguro(fila, "tipo_pregunta") or valor_seguro(fila, "complejidad_razonamiento") or valor_seguro(fila, "distribucion_evidencia"):
        documento.add_paragraph(
            f"Tipo: {valor_seguro(fila, 'tipo_pregunta')} | "
            f"Complejidad: {valor_seguro(fila, 'complejidad_razonamiento')} | "
            f"Evidencia: {valor_seguro(fila, 'distribucion_evidencia')}"
        )

    p = documento.add_paragraph()
    p.add_run("Respuesta generada:\n").bold = True
    p.add_run(str(fila["respuesta_generada"]))

    documento.add_paragraph("Puntuación de relevancia de la respuesta (0,0-1,0): ____________________")
    documento.add_paragraph("Comentarios de la experta:")
    documento.add_paragraph("______________________________________________________________________")
    documento.add_paragraph("______________________________________________________________________")
    documento.add_paragraph("______________________________________________________________________")

    documento.add_page_break()

documento.save(RUTA_WORD_EXPERTA)

print("Word guardado en:")
print(RUTA_WORD_EXPERTA)


Word guardado en:
/content/evaluacion_experta_rag_vectorial.docx


In [34]:
# Descargar archivos de salida desde Colab

from google.colab import files

archivos_salida = [
    RUTA_SALIDA_JSON,
    RUTA_SALIDA_JSON_METRICAS,
    RUTA_SALIDA_CSV_METRICAS,
    RUTA_SALIDA_CSV_RESUMIDO,
    RUTA_EVALUACION_EXPERTA,
    RUTA_WORD_EXPERTA
]

for ruta in archivos_salida:
    if os.path.exists(ruta):
        print("Descargando:", ruta)
        files.download(ruta)
    else:
        print("No existe todavía:", ruta)


Descargando: /content/resultados_rag_vectorial.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descargando: /content/resultados_rag_vectorial_con_evaluacion_automatica.json


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descargando: /content/evaluacion_experta_rag_vectorial.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>